In [1]:
from unsloth import FastLanguageModel
import torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/diogenes/pylingual_colaboration/pylingual_download/code/.pyllmpatch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
_MODEL = None
_TOKENIZER = None

def load_model_once(
    *,
    model_path: str,
    device_map: str = "auto",
    max_tokens: int
):
    """
    Load the merged model and tokenizer exactly once.
    """
    global _MODEL, _TOKENIZER

    if _MODEL is not None and _TOKENIZER is not None:
        return _MODEL, _TOKENIZER


    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=max_tokens + 8192,
        dtype=None,
        load_in_4bit=True,
        device_map=device_map,
    )

    # model.eval()
    FastLanguageModel.for_inference(model)

    _MODEL = model
    _TOKENIZER = tokenizer

    return _MODEL, _TOKENIZER


In [3]:
messages = [
        {
            "role": "system",
            "content": "You are a python programmer. Answer the question as best as you can. If you don't know the answer, say you don't know.",
        },
        {
            "role": "user",
            "content": "Write a python function that takes in a list of numbers and returns the sum of the numbers.",
        },
    ]

In [7]:
model, tokenizer = load_model_once(model_path="Qwen/Qwen2.5-Coder-7B-Instruct", max_tokens=16384)

In [10]:
CHAT_TEMPLATE = """{% for message in messages %}
{% if message['role'] == 'system' %}<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}<|im_start|>assistant
{{ message['content'] }}<|im_end|>
{% endif %}
{% endfor %}
{% if add_generation_prompt %}<|im_start|>assistant
{% endif %}"""

In [11]:
tokenizer.chat_template = CHAT_TEMPLATE

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    use_cache=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(response)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/diogenes/pylingual_colaboration/pylingual_download/code/.pyllmpatch/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/diogenes/pylingual_colaboration/pylingual_download/code/.pyllmpatch/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please 

Here's a simple Python function that takes in a list of numbers and returns the sum of the numbers:

```python
def sum_numbers(numbers):
    return sum(numbers)
```

You can use this function by passing in a list of numbers as an argument, like this:

```python
numbers = [1, 2, 3, 4, 5]
result = sum_numbers(numbers)
print(result)  # Output: 15
```

This function uses the built-in `sum()` function in Python to calculate the sum of the numbers in the list. It then returns the result.<|file_sep|><|fim_prefix|>/README.md
# Python Function to Calculate the Sum of a List of Numbers

This repository contains a Python function that calculates the sum of a list of numbers.

## Usage

To use the function, you can import it into your Python script and call it with a list of numbers as an argument. Here's an example:

```python
from sum_numbers import sum_numbers

numbers = [1, 2, 3, 4, 5]
result = sum_numbers(numbers)
print(result)  # Output: 15
```

The `sum_numbers` function takes a list of num